# HybridGraphFNet — NeuroGraph HCP-Gender

**Task:** Graph-level binary classification (Male vs Female) on brain functional connectomes.  
**Head:** GatedPooling → graph vector → MLP classifier (same as Peptides-func).  
**Eigenbasis:** Precomputed offline, cached to disk — no `eigh` during training.

| Property | Detail |
|---|---|
| Dataset | NeuroGraph HCP-Gender |
| Graphs | 1,078 brain connectomes |
| Nodes/graph | 200 (Schaefer-200 parcellation, fixed) |
| Node features | fMRI correlation vectors |
| Classes | 2 (Male / Female) |
| Metric | Accuracy + Macro F1 |
| Split | Manual 80/10/10 stratified |

In [3]:
!pip install torch_geometric

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import time, os, gc
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import NeuroGraphDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def set_seed(s):
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    np.random.seed(s); torch.backends.cudnn.deterministic = True

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 1. Load NeuroGraph HCP-Gender & Inspect

In [5]:
DATA_ROOT = './data'
CACHE_DIR = './eigenbasis_cache_hcp_gender'
TRUNC_K   = 64

# NeuroGraphDataset does NOT have a built-in split parameter
full_dataset = NeuroGraphDataset(root=DATA_ROOT, name='HCPGender')

NUM_CLASSES = full_dataset.num_classes if hasattr(full_dataset, 'num_classes') else 2

# Inspect node features
sample = full_dataset[0]
HAS_FEATURES = sample.x is not None
NODE_FEAT_DIM = sample.x.shape[-1] if HAS_FEATURES else 1
print(f'Node features: {"dim=" + str(NODE_FEAT_DIM) if HAS_FEATURES else "ABSENT"}')

# Edge features
HAS_EDGE_ATTR = sample.edge_attr is not None
EDGE_DIM = sample.edge_attr.shape[-1] if (HAS_EDGE_ATTR and sample.edge_attr.dim() > 1) else (1 if HAS_EDGE_ATTR else 0)
print(f'Edge attr: {"dim=" + str(EDGE_DIM) if HAS_EDGE_ATTR else "absent"}')

# Node/edge counts
all_n = [d.num_nodes for d in full_dataset]
all_e = [d.num_edges for d in full_dataset]
print(f'\nDataset: {len(full_dataset)} graphs, {NUM_CLASSES} classes')
print(f'Nodes: min={min(all_n)}, max={max(all_n)}, mean={np.mean(all_n):.1f}')
print(f'Edges: min={min(all_e)}, max={max(all_e)}, mean={np.mean(all_e):.0f}')

# Class distribution
all_labels = []
for d in full_dataset:
    y = d.y.item() if d.y.dim() == 0 else d.y[0].item()
    all_labels.append(int(y))
all_labels = np.array(all_labels)

print(f'\nClass distribution:')
for c in range(NUM_CLASSES):
    n = (all_labels == c).sum()
    print(f'  Class {c}: {n:>5d} ({100*n/len(all_labels):.1f}%)')

Extracting data/HCPGender/raw/r6hlz2arm7yiy6v6981cv2nzq3b0meax.zip
Processing...
Done!


Node features: dim=1000
Edge attr: absent

Dataset: 1078 graphs, 2 classes
Nodes: min=1000, max=1000, mean=1000.0
Edges: min=32940, max=49694, mean=45579

Class distribution:
  Class 0:   585 (54.3%)
  Class 1:   493 (45.7%)


## 2. Train/Val/Test Split (Stratified 80/10/10)

NeuroGraph HCP-Gender has no built-in split — we create one manually with stratification.

In [6]:
SPLIT_SEED = 42  # Fixed for reproducibility across seeds

indices = np.arange(len(full_dataset))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=SPLIT_SEED, stratify=all_labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SPLIT_SEED, stratify=all_labels[temp_idx])

train_ds = full_dataset[train_idx.tolist()]
val_ds   = full_dataset[val_idx.tolist()]
test_ds  = full_dataset[test_idx.tolist()]

print(f'Split sizes: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')

# Verify stratification
for name, ds in [('Train', train_ds), ('Val', val_ds), ('Test', test_ds)]:
    ys = [d.y.item() if d.y.dim()==0 else d.y[0].item() for d in ds]
    counts = np.bincount(ys, minlength=NUM_CLASSES)
    pcts = 100 * counts / len(ys)
    dist = ', '.join(f'C{i}={counts[i]}({pcts[i]:.0f}%)' for i in range(NUM_CLASSES))
    print(f'  {name}: {dist}')

Split sizes: Train=862, Val=108, Test=108
  Train: C0=468(54%), C1=394(46%)
  Val: C0=58(54%), C1=50(46%)
  Test: C0=59(55%), C1=49(45%)


## 3. Precompute Eigenbasis — One-Time Offline

200 nodes/graph → eigh is cheap (0.001s/graph). Disk estimate first.

In [7]:
trunc_bytes = sum(n * TRUNC_K * 4 for n in all_n)
full_bytes  = sum(n * n * 4 for n in all_n)
est_time = sum((n/500)**3 * 0.05 for n in all_n)

print('=== Disk Size Estimates ===')
print(f'Total graphs: {len(all_n):,}')
print(f'Truncated k={TRUNC_K}: {trunc_bytes/1e6:.1f} MB')
print(f'Full N×N: {full_bytes/1e6:.1f} MB')
print(f'Estimated precompute time: ~{est_time:.1f}s')
print(f'>>> N=200 everywhere — this is very fast and small.')

=== Disk Size Estimates ===
Total graphs: 1,078
Truncated k=64: 276.0 MB
Full N×N: 4312.0 MB
Estimated precompute time: ~431.2s
>>> N=200 everywhere — this is very fast and small.


In [8]:
def precompute_eigenbasis_for_split(dataset, split_name, cache_dir, k=64):
    split_dir = os.path.join(cache_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)
    eigh_failures = 0
    total_bytes = 0
    t_start = time.perf_counter()

    for idx in tqdm(range(len(dataset)), desc=f'Precompute {split_name}'):
        data = dataset[idx]
        n = data.num_nodes
        edge_index = data.edge_index
        adj = torch.zeros(n, n)
        adj[edge_index[0], edge_index[1]] = 1.0
        adj = adj + torch.eye(n)

        deg = adj.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt = torch.diag(deg_inv_sqrt)
        A_norm = D_inv_sqrt @ adj @ D_inv_sqrt
        L = torch.eye(n) - A_norm

        try:
            _, U = torch.linalg.eigh(L)
            max_abs_idx = torch.abs(U).argmax(dim=0)
            signs = torch.sign(U[max_abs_idx, torch.arange(U.size(1))])
            signs[signs == 0] = 1.0
            U = U * signs.unsqueeze(0)
        except Exception as e:
            eigh_failures += 1
            U = torch.eye(n)
            if eigh_failures <= 5:
                print(f'  WARNING: eigh failed graph {idx} (n={n}): {e}')

        k_actual = min(k, n)
        U_trunc = U[:, :k_actual]
        save_path = os.path.join(split_dir, f'{idx}.pt')
        torch.save({'U': U_trunc.clone(), 'n': n, 'k': k_actual}, save_path)
        total_bytes += os.path.getsize(save_path)

    elapsed = time.perf_counter() - t_start
    print(f'  {split_name}: {len(dataset)} graphs, {eigh_failures} failures, '
          f'{elapsed:.1f}s, {total_bytes/1e6:.1f} MB')
    return {'split': split_name, 'eigh_failures': eigh_failures, 'time_s': elapsed, 'disk_mb': total_bytes/1e6}

In [9]:
for split_name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    split_dir = os.path.join(CACHE_DIR, split_name)
    expected = len(ds)
    existing = len([f for f in os.listdir(split_dir) if f.endswith('.pt')]) if os.path.isdir(split_dir) else 0
    if existing >= expected:
        print(f'{split_name}: cache exists ({existing} files), skipping')
    else:
        precompute_eigenbasis_for_split(ds, split_name, CACHE_DIR, k=TRUNC_K)

print('Eigenbasis cache ready.')

Precompute train:  53%|█████▎    | 460/862 [00:37<00:34, 11.58it/s]

Precompute train:  64%|██████▍   | 552/862 [00:44<00:24, 12.71it/s]

Precompute train:  76%|███████▌  | 654/862 [00:53<00:17, 11.75it/s]

Precompute train: 100%|██████████| 862/862 [01:09<00:00, 12.34it/s]


  train: 862 graphs, 3 failures, 69.8s, 222.0 MB


Precompute val: 100%|██████████| 108/108 [00:08<00:00, 12.46it/s]


  val: 108 graphs, 0 failures, 8.7s, 27.8 MB


Precompute test:  48%|████▊     | 52/108 [00:04<00:04, 12.54it/s]

Precompute test: 100%|██████████| 108/108 [00:08<00:00, 12.36it/s]

  test: 108 graphs, 1 failures, 8.7s, 27.8 MB
Eigenbasis cache ready.


## 4. Cached Dataset Wrapper & DataLoaders

In [10]:
class CachedEigenbasisDataset:
    def __init__(self, base_dataset, cache_dir, split_name, k_trunc=64):
        self.base = base_dataset
        self.split_dir = os.path.join(cache_dir, split_name)
        self.k_trunc = k_trunc

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        data = self.base[idx].clone()
        cache = torch.load(os.path.join(self.split_dir, f'{idx}.pt'), weights_only=True)
        U = cache['U']
        if U.size(1) < self.k_trunc:
            U = F.pad(U, (0, self.k_trunc - U.size(1)))
        data.cached_U = U

        # Ensure x exists
        if data.x is None:
            from torch_geometric.utils import degree
            row = data.edge_index[0]
            data.x = degree(row, num_nodes=data.num_nodes).float().unsqueeze(-1)
        return data


def make_loaders(batch_size=32):
    tl = DataLoader(CachedEigenbasisDataset(train_ds, CACHE_DIR, 'train', TRUNC_K), batch_size=batch_size, shuffle=True)
    vl = DataLoader(CachedEigenbasisDataset(val_ds, CACHE_DIR, 'val', TRUNC_K), batch_size=batch_size, shuffle=False)
    tel = DataLoader(CachedEigenbasisDataset(test_ds, CACHE_DIR, 'test', TRUNC_K), batch_size=batch_size, shuffle=False)
    return tl, vl, tel

# Quick sanity check
_tl, _, _ = make_loaders(4)
_b = next(iter(_tl))
print(f'Batch: x={_b.x.shape}, cached_U={_b.cached_U.shape}, y={_b.y.shape}')
del _tl, _b

Batch: x=torch.Size([4000, 1000]), cached_U=torch.Size([4000, 64]), y=torch.Size([4])


## 5. Model Components (backbone identical to Peptides)

In [11]:
class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=0):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1) if edge_dim > 0 else None
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        if edge_attr_dense is not None and self.edge_lin is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))


class SpectralMixMH(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1, 2), x)
        fil        = torch.sigmoid(self.filter_gen(x_hat))
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)
        x_out      = x_out * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))


class GatedPooling(nn.Module):
    """Graph-level pooling via learned gating (same as Peptides pipeline)."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, mask):
        scores  = self.gate(x).squeeze(-1)
        scores  = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)  # [B, H]


print('Components defined: DenseGCNLayer, SpectralMixMH, GatedPooling')

Components defined: DenseGCNLayer, SpectralMixMH, GatedPooling


## 6. HybridGraphFNet — Graph-Level Head (GatedPooling + MLP)

Same backbone as Peptides. GatedPooling → graph vector → binary classifier.  
**NOT** node-level.

In [12]:
class HybridGraphFNet_GraphLevel(nn.Module):
    def __init__(self, in_dim=200, hidden_dim=128, num_layers=4, num_classes=2,
                 num_heads=4, lap_k=8, dropout=0.1, edge_dim=0):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)

        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
        )
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'local':  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                'global': SpectralMixMH(hidden_dim, num_heads=num_heads),
                'gate':   nn.Linear(hidden_dim, hidden_dim),
                'norm':   nn.LayerNorm(hidden_dim),
            }) for _ in range(num_layers)
        ])

        self.pool = GatedPooling(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes),
        )

    def compute_A_norm(self, adj, mask):
        B, N, _ = adj.shape
        A_list = []
        for b in range(B):
            n = int(mask[b].sum().item())
            adj_b = adj[b, :n, :n]
            deg = adj_b.sum(dim=1)
            deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
            D_inv_sqrt = torch.diag(deg_inv_sqrt)
            A_norm_b = D_inv_sqrt @ adj_b @ D_inv_sqrt
            A_list.append(F.pad(A_norm_b, (0, N-n, 0, N-n)))
        return torch.stack(A_list)

    def forward(self, data):
        x, mask = to_dense_batch(data.x.float(), data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))

        if hasattr(data, 'edge_attr') and data.edge_attr is not None:
            ea = data.edge_attr.float()
            if ea.dim() == 1: ea = ea.unsqueeze(-1)
            edge_attr_dense = to_dense_adj(data.edge_index, data.batch, edge_attr=ea, max_num_nodes=x.size(1))
        else:
            edge_attr_dense = None

        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        A_norm = self.compute_A_norm(adj, mask)

        U, _ = to_dense_batch(data.cached_U.float(), data.batch)
        U = U * mask.unsqueeze(-1)

        x = self.input_proj(x)
        k = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        x = x + self.pe_encoder(lap_pe)

        for layer in self.layers:
            x_res    = x
            x_local  = layer['local'](x, A_norm, edge_attr_dense)
            x_global = layer['global'](x, U, mask)
            gate     = torch.sigmoid(layer['gate'](x))
            x_mix    = gate * x_local + (1 - gate) * x_global
            x        = layer['norm'](x_res + self.dropout(x_mix))

        x = x * mask.unsqueeze(-1)
        graph_emb = self.pool(x, mask)
        return self.classifier(graph_emb)


# Sanity check
set_seed(0)
model = HybridGraphFNet_GraphLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, edge_dim=EDGE_DIM,
).to(device)

print(f'Parameters: {count_params(model):,}')
_tl, _, _ = make_loaders(4)
_b = next(iter(_tl)).to(device)
with torch.no_grad():
    logits = model(_b)
print(f'Forward pass OK: logits={logits.shape}')
del _tl, _b

Parameters: 438,147
Forward pass OK: logits=torch.Size([4, 2])


## 7. Evaluation: Graph-Level Accuracy + Macro F1

In [13]:
def evaluate_graph_level(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            preds = logits.argmax(dim=-1).cpu()
            labels = batch.y.cpu()
            if labels.dim() > 1: labels = labels.squeeze(-1)
            all_preds.append(preds)
            all_labels.append(labels)
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, f1

print('evaluate_graph_level defined')

evaluate_graph_level defined


## 8. Gate Health Check

In [14]:
def check_gate_health(model, val_loader, device):
    model.eval()
    batch = next(iter(val_loader)).to(device)
    with torch.no_grad():
        x, mask = to_dense_batch(batch.x.float(), batch.batch)
        U, _ = to_dense_batch(batch.cached_U.float(), batch.batch)
        U = U * mask.unsqueeze(-1)
        x_enc = model.input_proj(x)
        k = min(model.lap_k, U.size(-1))
        x_enc = x_enc + model.pe_encoder(U[:, :, :k] * mask.unsqueeze(-1))

        print('\n--- Gate Health Check ---')
        collapsed = False
        for i, layer in enumerate(model.layers):
            gv = torch.sigmoid(layer['gate'](x_enc))
            mg, sg = gv.mean().item(), gv.std().item()
            if mg > 0.85:   status = 'COLLAPSED->GCN';     collapsed = True
            elif mg < 0.15: status = 'COLLAPSED->SPECTRAL'; collapsed = True
            elif sg < 0.05: status = 'UNIFORM';             collapsed = True
            else:           status = 'HEALTHY'
            print(f'  Layer {i} | mean={mg:.4f} | std={sg:.4f} | {status}')
        if collapsed: print('  ACTION: lr will be reset.')
        else:         print('  All gates healthy.')
        print('-------------------------\n')
    model.train()
    return collapsed

## 9. Timing Comparison: Live eigh vs Precomputed

Second real-dataset data point for the paper's efficiency story (after PascalVOC-SP).

In [15]:
def compute_laplacian_basis_LIVE(adj, mask):
    B, N, _ = adj.shape
    A_list, U_list = [], []
    for b in range(B):
        n = int(mask[b].sum().item())
        adj_b = adj[b, :n, :n]
        deg = adj_b.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt = torch.diag(deg_inv_sqrt)
        A_norm_b = D_inv_sqrt @ adj_b @ D_inv_sqrt
        L_b = torch.eye(n, device=adj.device) - A_norm_b
        try:
            _, U_b = torch.linalg.eigh(L_b)
        except Exception:
            U_b = torch.eye(n, device=adj.device)
        A_list.append(F.pad(A_norm_b, (0, N-n, 0, N-n)))
        U_list.append(F.pad(U_b, (0, N-n, 0, N-n)))
    return torch.stack(A_list), torch.stack(U_list)

NUM_TIMING_BATCHES = 10
BS_TIMING = 16
_tl, _, _ = make_loaders(BS_TIMING)
_it = iter(_tl)
_batches = [next(_it).to(device) for _ in range(min(NUM_TIMING_BATCHES, len(_tl)))]

model_timing = HybridGraphFNet_GraphLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4, num_classes=NUM_CLASSES, edge_dim=EDGE_DIM,
).to(device).eval()

# OLD: live eigh per batch
t_old = 0.0
with torch.no_grad():
    for b in _batches:
        x, mask = to_dense_batch(b.x.float(), b.batch)
        adj = to_dense_adj(b.edge_index, b.batch, max_num_nodes=x.size(1))
        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        t0 = time.perf_counter()
        compute_laplacian_basis_LIVE(adj, mask)
        if device.type == 'cuda': torch.cuda.synchronize()
        t_old += time.perf_counter() - t0

# NEW: precomputed U lookup
t_new = 0.0
with torch.no_grad():
    for b in _batches:
        x, mask = to_dense_batch(b.x.float(), b.batch)
        adj = to_dense_adj(b.edge_index, b.batch, max_num_nodes=x.size(1))
        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        t0 = time.perf_counter()
        model_timing.compute_A_norm(adj, mask)
        to_dense_batch(b.cached_U.float(), b.batch)
        if device.type == 'cuda': torch.cuda.synchronize()
        t_new += time.perf_counter() - t0

nb = len(_batches)
print(f'=== TIMING COMPARISON ({nb} batches, BS={BS_TIMING}) ===')
print(f'OLD (live eigh):     {t_old/nb*1000:.1f} ms/batch')
print(f'NEW (precomputed U): {t_new/nb*1000:.1f} ms/batch')
print(f'Speedup:             {t_old/max(t_new,1e-6):.1f}x')
del model_timing, _batches, _tl

=== TIMING COMPARISON (10 batches, BS=16) ===
OLD (live eigh):     328.7 ms/batch
NEW (precomputed U): 17.5 ms/batch
Speedup:             18.8x


## 10. Memory & Timing Smoke Test

Run BEFORE full training. N=200 should be very comfortable on any GPU.

In [16]:
BATCH_SIZE = 32

_tl, _, _ = make_loaders(BATCH_SIZE)
_model = HybridGraphFNet_GraphLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4, num_classes=NUM_CLASSES, edge_dim=EDGE_DIM,
).to(device)
_model.train()
_opt = optim.AdamW(_model.parameters(), lr=1e-3)
_crit = nn.CrossEntropyLoss()

if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()

_b = next(iter(_tl)).to(device)
t0 = time.perf_counter()
logits = _model(_b)
y = _b.y; y = y.squeeze(-1) if y.dim() > 1 else y
loss = _crit(logits, y.long())
loss.backward()
torch.nn.utils.clip_grad_norm_(_model.parameters(), 1.0)
_opt.step(); _opt.zero_grad()
if device.type == 'cuda': torch.cuda.synchronize()
t_batch = time.perf_counter() - t0

peak_gb = torch.cuda.max_memory_allocated()/1e9 if device.type == 'cuda' else 0.0
total_gb = torch.cuda.get_device_properties(0).total_memory/1e9 if device.type == 'cuda' else 16.0
batches_ep = len(train_ds) // BATCH_SIZE

print('=== MEMORY & TIMING SMOKE TEST ===')
print(f'BS={BATCH_SIZE}, per-batch: {t_batch*1000:.0f} ms')
print(f'Peak VRAM: {peak_gb:.2f} GB / {total_gb:.1f} GB ({100*peak_gb/total_gb:.0f}%)')
print(f'Est. time/epoch: {t_batch*batches_ep:.1f}s, 200 epochs: {t_batch*batches_ep*200/3600:.1f}h')
del _model, _opt, _b, _tl

=== MEMORY & TIMING SMOKE TEST ===
BS=32, per-batch: 328 ms
Peak VRAM: 1.69 GB / 15.6 GB (11%)
Est. time/epoch: 8.5s, 200 epochs: 0.5h


## 11. Trivial Majority Baseline

Floor to compare against — binary classification, balanced or not.

In [17]:
# Majority class from training set
train_y = np.array([d.y.item() if d.y.dim()==0 else d.y[0].item() for d in train_ds])
majority_class = int(np.bincount(train_y).argmax())
majority_pct = 100 * (train_y == majority_class).sum() / len(train_y)

test_y = np.array([d.y.item() if d.y.dim()==0 else d.y[0].item() for d in test_ds])
maj_preds = np.full_like(test_y, majority_class)
maj_acc = accuracy_score(test_y, maj_preds)
maj_f1  = f1_score(test_y, maj_preds, average='macro', zero_division=0)

print(f'=== TRIVIAL MAJORITY BASELINE ===')
print(f'Majority class: {majority_class} ({majority_pct:.1f}% of train)')
print(f'Test accuracy (majority): {maj_acc:.4f}')
print(f'Test macro F1 (majority): {maj_f1:.4f}')
print(f'Model must beat these numbers.')

=== TRIVIAL MAJORITY BASELINE ===
Majority class: 0 (54.3% of train)
Test accuracy (majority): 0.5463
Test macro F1 (majority): 0.3533
Model must beat these numbers.


## 12. Training Loop (Kaggle Background Compatible)

Checkpointing every 5 epochs. Single-seed per session.  
After seed 0: auto-diagnostic vs majority baseline.

In [18]:
CKPT_EVERY = 5

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_val_acc, best_epoch, epochs_no_improve, gate_checked):
    torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(), 'epoch': epoch, 'best_val_acc': best_val_acc,
                'best_epoch': best_epoch, 'epochs_no_improve': epochs_no_improve, 'gate_checked': gate_checked}, path)

def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, weights_only=False)
    model.load_state_dict(ckpt['model_state']); optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    return ckpt['epoch'], ckpt['best_val_acc'], ckpt['best_epoch'], ckpt['epochs_no_improve'], ckpt['gate_checked']


def train_single_seed(
    seed, in_dim=200, hidden_dim=128, num_layers=4, num_classes=2,
    num_heads=4, lap_k=8, edge_dim=0,
    max_epochs=200, patience=30, batch_size=32, lr=1e-3,
):
    train_loader, val_loader, test_loader = make_loaders(batch_size)

    # Class weights for binary CE
    train_y = torch.tensor([d.y.item() if d.y.dim()==0 else d.y[0].item() for d in train_ds])
    cc = torch.bincount(train_y, minlength=num_classes).float().clamp(min=1)
    cw = (1.0 / cc); cw = (cw / cw.sum() * num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)

    best_ckpt   = f'best_model_hcp_gender_seed{seed}.pt'
    resume_ckpt = f'resume_hcp_gender_seed{seed}.pt'

    print('=' * 65)
    print(f'HCP-Gender | Seed {seed} | BS={batch_size}')
    print('=' * 65)

    set_seed(seed)
    model = HybridGraphFNet_GraphLevel(
        in_dim=in_dim, hidden_dim=hidden_dim, num_layers=num_layers,
        num_classes=num_classes, num_heads=num_heads, lap_k=lap_k, edge_dim=edge_dim,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-5)
    print(f'Parameters: {count_params(model):,}')

    start_epoch = 1; best_val_acc = 0.0; best_epoch = 0; epochs_no_improve = 0; gate_checked = False

    if os.path.exists(resume_ckpt):
        print(f'\n>>> RESUMING from {resume_ckpt}')
        prev_ep, best_val_acc, best_epoch, epochs_no_improve, gate_checked = \
            load_checkpoint(resume_ckpt, model, optimizer, scheduler)
        start_epoch = prev_ep + 1
        print(f'    Resuming at epoch {start_epoch}')
    else:
        print('No checkpoint — starting fresh.')

    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    for epoch in range(start_epoch, max_epochs + 1):
        if epoch == 6 and not gate_checked:
            gc_result = check_gate_health(model, val_loader, device)
            gate_checked = True
            if gc_result:
                optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs-epoch, eta_min=1e-5)

        model.train()
        total_loss = 0; optimizer.zero_grad()
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f'Seed {seed} | Ep {epoch}/{max_epochs}', leave=False)

        for step, batch in pbar:
            batch = batch.to(device)
            logits = model(batch)
            y = batch.y; y = y.squeeze(-1) if y.dim() > 1 else y
            loss = criterion(logits, y.long())
            loss.backward()
            total_loss += loss.item()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad()
            pbar.set_postfix({'loss': f'{total_loss/(step+1):.4f}'})

        scheduler.step()

        val_acc, val_f1 = evaluate_graph_level(model, val_loader, device)
        improved = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc; best_epoch = epoch; epochs_no_improve = 0
            torch.save(model.state_dict(), best_ckpt); improved = ' *BEST*'
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or improved:
            print(f'  Ep {epoch:>3d} | loss={total_loss/len(train_loader):.4f} | '
                  f'val_acc={val_acc:.4f} val_F1={val_f1:.4f} | best={best_val_acc:.4f}@ep{best_epoch}{improved}')

        if epoch % CKPT_EVERY == 0:
            save_checkpoint(resume_ckpt, model, optimizer, scheduler,
                           epoch, best_val_acc, best_epoch, epochs_no_improve, gate_checked)

        if epochs_no_improve >= patience:
            print(f'  Early stopping at epoch {epoch}'); break

    save_checkpoint(resume_ckpt, model, optimizer, scheduler,
                   epoch, best_val_acc, best_epoch, epochs_no_improve, gate_checked)

    elapsed = time.time() - start_time
    model.load_state_dict(torch.load(best_ckpt, weights_only=True))
    test_acc, test_f1 = evaluate_graph_level(model, test_loader, device)
    peak_mem = torch.cuda.max_memory_allocated()/1e9 if device.type == 'cuda' else 0.0

    print(f'\n  SEED {seed}: test_acc={test_acc:.4f} test_F1={test_f1:.4f} (val={best_val_acc:.4f}@ep{best_epoch})')
    print(f'  Time: {elapsed:.0f}s | Peak VRAM: {peak_mem:.2f} GB')

    # ---- Diagnostic vs majority baseline ----
    print(f'\n  DIAGNOSTIC:')
    print(f'    Majority baseline: acc={maj_acc:.4f}, F1={maj_f1:.4f}')
    d_acc, d_f1 = test_acc - maj_acc, test_f1 - maj_f1
    print(f'    Model vs majority: acc {d_acc:+.4f}, F1 {d_f1:+.4f}')
    if d_acc < 0.02 and d_f1 < 0.02:
        print(f'    *** WARNING: At/below majority baseline — investigate before proceeding. ***')
    else:
        print(f'    Model exceeds majority baseline — safe to proceed to more seeds.')

    result = {'seed': seed, 'test_acc': test_acc, 'test_f1': test_f1,
              'best_val_acc': best_val_acc, 'best_epoch': best_epoch,
              'time_s': elapsed, 'peak_gb': peak_mem}

    import csv
    csv_path = 'results_hcp_gender.csv'
    file_exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(result.keys()))
        if not file_exists: writer.writeheader()
        writer.writerow(result)
    print(f'  Result appended to {csv_path}')
    return result

## 13. Run Training (Single Seed)

Set `RUN_SEED` below. Run seed 0 first — verify diagnostic, then proceed.

In [19]:
# ============================
# SET THIS PER KAGGLE SESSION
RUN_SEED = 0
# ============================

result = train_single_seed(
    seed=RUN_SEED,
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, num_heads=4, lap_k=8, edge_dim=EDGE_DIM,
    max_epochs=200, patience=30, batch_size=32, lr=1e-3,
)

HCP-Gender | Seed 0 | BS=32
Parameters: 438,147
No checkpoint — starting fresh.


  Ep   1 | loss=0.6900 | val_acc=0.5093 val_F1=0.5072 | best=0.5093@ep1 *BEST*


  Ep   2 | loss=0.6819 | val_acc=0.5556 val_F1=0.5282 | best=0.5556@ep2 *BEST*


  Ep   5 | loss=0.6400 | val_acc=0.6111 val_F1=0.6110 | best=0.6111@ep5 *BEST*

--- Gate Health Check ---
  Layer 0 | mean=0.5029 | std=0.1031 | HEALTHY
  Layer 1 | mean=0.5054 | std=0.0922 | HEALTHY
  Layer 2 | mean=0.5037 | std=0.0919 | HEALTHY
  Layer 3 | mean=0.5082 | std=0.0896 | HEALTHY
  All gates healthy.
-------------------------



  Ep   6 | loss=0.6342 | val_acc=0.6667 val_F1=0.6073 | best=0.6667@ep6 *BEST*


...

  Ep  30 | loss=0.2832 | val_acc=0.6667 val_F1=0.6003 | best=0.8704@ep17


  Ep  35 | loss=0.0804 | val_acc=0.8333 val_F1=0.8324 | best=0.8704@ep17


  Ep  40 | loss=0.2425 | val_acc=0.7870 val_F1=0.7787 | best=0.8704@ep17


  Ep  45 | loss=0.0242 | val_acc=0.7963 val_F1=0.7937 | best=0.8704@ep17


  Early stopping at epoch 47

  SEED 0: test_acc=0.8333 test_F1=0.8313 (val=0.8704@ep17)
  Time: 342s | Peak VRAM: 1.87 GB

  DIAGNOSTIC:
    Majority baseline: acc=0.5463, F1=0.3533
    Model vs majority: acc +0.2870, F1 +0.4780
    Model exceeds majority baseline — safe to proceed to more seeds.
  Result appended to results_hcp_gender.csv


## 14. VRAM Scaling — Real-Dataset Confirmation

Second real-dataset data point for the paper's efficiency claim (after PascalVOC-SP).

In [22]:
if device.type == 'cuda':
    peak_vram = torch.cuda.max_memory_allocated() / 1e9
else:
    peak_vram = 0.0

print(f'=== VRAM SCALING — REAL-DATASET CONFIRMATION ===')
print(f'Dataset: HCP-Gender (N=200, 1,078 graphs)')
print(f'Peak VRAM during training: {peak_vram:.2f} GB')
print(f'\nComparison with other datasets:')
print(f'  Peptides (N~150):      peak VRAM in paper')
print(f'  PascalVOC-SP (N~479):  0.15 GB peak VRAM')
print(f'  HCP-Gender (N=1000):    {peak_vram:.2f} GB peak VRAM')

=== VRAM SCALING — REAL-DATASET CONFIRMATION ===
Dataset: HCP-Gender (N=200, 1,078 graphs)
Peak VRAM during training: 1.87 GB

Comparison with other datasets:
  Peptides (N~150):      peak VRAM in paper
  PascalVOC-SP (N~479):  0.15 GB peak VRAM
  HCP-Gender (N=1000):    1.87 GB peak VRAM


## 15. Results Summary

In [21]:
import csv

csv_path = 'results_hcp_gender.csv'
all_results = []
if os.path.exists(csv_path):
    with open(csv_path, 'r') as f:
        for row in csv.DictReader(f):
            all_results.append({k: float(v) if k != 'seed' else int(float(v)) for k, v in row.items()})

if all_results:
    accs = [r['test_acc'] for r in all_results]
    f1s  = [r['test_f1'] for r in all_results]
    n = len(all_results)
    print(f'HCP-Gender | HybridGraphFNet (Graph-Level, GatedPooling)')
    for r in all_results:
        print(f'  Seed {r["seed"]}: acc={r["test_acc"]:.4f} F1={r["test_f1"]:.4f} (ep {r["best_epoch"]:.0f})')
    if n >= 2:
        print(f'  Mean acc: {np.mean(accs):.4f} +/- {np.std(accs):.4f}')
        print(f'  Mean F1:  {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}')
    print(f'  Majority baseline: acc={maj_acc:.4f} F1={maj_f1:.4f}')
else:
    print('No results found. Run training first.')

HCP-Gender | HybridGraphFNet (Graph-Level, GatedPooling)
  Seed 0: acc=0.8333 F1=0.8313 (ep 17)
  Majority baseline: acc=0.5463 F1=0.3533


In [23]:
# ============================
# SET THIS PER KAGGLE SESSION
RUN_SEED = 1
# ============================

result = train_single_seed(
    seed=RUN_SEED,
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, num_heads=4, lap_k=8, edge_dim=EDGE_DIM,
    max_epochs=200, patience=30, batch_size=32, lr=1e-3,
)

HCP-Gender | Seed 1 | BS=32
Parameters: 438,147
No checkpoint — starting fresh.


  Ep   1 | loss=0.6923 | val_acc=0.5556 val_F1=0.5282 | best=0.5556@ep1 *BEST*


  Ep   2 | loss=0.6824 | val_acc=0.5648 val_F1=0.5584 | best=0.5648@ep2 *BEST*


  Ep   3 | loss=0.6754 | val_acc=0.5741 val_F1=0.5479 | best=0.5741@ep3 *BEST*


  Ep   4 | loss=0.6519 | val_acc=0.7315 val_F1=0.7262 | best=0.7315@ep4 *BEST*


...

  Ep  50 | loss=0.0193 | val_acc=0.8519 val_F1=0.8516 | best=0.9074@ep38


  Ep  55 | loss=0.0045 | val_acc=0.8704 val_F1=0.8704 | best=0.9074@ep38


  Ep  60 | loss=0.0088 | val_acc=0.8333 val_F1=0.8328 | best=0.9074@ep38


  Ep  65 | loss=0.0000 | val_acc=0.8796 val_F1=0.8796 | best=0.9074@ep38


  Early stopping at epoch 68

  SEED 1: test_acc=0.8148 test_F1=0.8138 (val=0.9074@ep38)
  Time: 497s | Peak VRAM: 1.87 GB

  DIAGNOSTIC:
    Majority baseline: acc=0.5463, F1=0.3533
    Model vs majority: acc +0.2685, F1 +0.4605
    Model exceeds majority baseline — safe to proceed to more seeds.
  Result appended to results_hcp_gender.csv


In [25]:
# ============================
# SET THIS PER KAGGLE SESSION
RUN_SEED = 2
# ============================

result = train_single_seed(
    seed=RUN_SEED,
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, num_heads=4, lap_k=8, edge_dim=EDGE_DIM,
    max_epochs=200, patience=30, batch_size=32, lr=1e-3,
)

HCP-Gender | Seed 2 | BS=32
Parameters: 438,147

>>> RESUMING from resume_hcp_gender_seed2.pt
    Resuming at epoch 6

--- Gate Health Check ---
  Layer 0 | mean=0.5120 | std=0.0787 | HEALTHY
  Layer 1 | mean=0.4941 | std=0.0554 | HEALTHY
  Layer 2 | mean=0.4964 | std=0.0519 | HEALTHY
  Layer 3 | mean=0.5036 | std=0.0516 | HEALTHY
  All gates healthy.
-------------------------



  Ep   6 | loss=0.6489 | val_acc=0.7870 val_F1=0.7870 | best=0.7870@ep6 *BEST*


  Ep   9 | loss=0.4694 | val_acc=0.8241 val_F1=0.8228 | best=0.8241@ep9 *BEST*


  Ep  10 | loss=0.4701 | val_acc=0.7870 val_F1=0.7802 | best=0.8241@ep9


  Ep  12 | loss=0.3431 | val_acc=0.8519 val_F1=0.8518 | best=0.8519@ep12 *BEST*


...

  Ep  50 | loss=0.0422 | val_acc=0.8333 val_F1=0.8319 | best=0.8704@ep37


  Ep  55 | loss=0.0802 | val_acc=0.8519 val_F1=0.8510 | best=0.8704@ep37


  Ep  60 | loss=0.0691 | val_acc=0.7500 val_F1=0.7358 | best=0.8704@ep37


  Ep  65 | loss=0.0666 | val_acc=0.7315 val_F1=0.7209 | best=0.8704@ep37


  Early stopping at epoch 67

  SEED 2: test_acc=0.7870 test_F1=0.7848 (val=0.8704@ep37)
  Time: 455s | Peak VRAM: 2.53 GB

  DIAGNOSTIC:
    Majority baseline: acc=0.5463, F1=0.3533
    Model vs majority: acc +0.2407, F1 +0.4315
    Model exceeds majority baseline — safe to proceed to more seeds.
  Result appended to results_hcp_gender.csv


In [26]:
import csv

csv_path = 'results_hcp_gender.csv'
all_results = []
if os.path.exists(csv_path):
    with open(csv_path, 'r') as f:
        for row in csv.DictReader(f):
            all_results.append({k: float(v) if k != 'seed' else int(float(v)) for k, v in row.items()})

if all_results:
    accs = [r['test_acc'] for r in all_results]
    f1s  = [r['test_f1'] for r in all_results]
    n = len(all_results)
    print(f'HCP-Gender | HybridGraphFNet (Graph-Level, GatedPooling)')
    for r in all_results:
        print(f'  Seed {r["seed"]}: acc={r["test_acc"]:.4f} F1={r["test_f1"]:.4f} (ep {r["best_epoch"]:.0f})')
    if n >= 2:
        print(f'  Mean acc: {np.mean(accs):.4f} +/- {np.std(accs):.4f}')
        print(f'  Mean F1:  {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}')
    print(f'  Majority baseline: acc={maj_acc:.4f} F1={maj_f1:.4f}')
else:
    print('No results found. Run training first.')

HCP-Gender | HybridGraphFNet (Graph-Level, GatedPooling)
  Seed 0: acc=0.8333 F1=0.8313 (ep 17)
  Seed 1: acc=0.8148 F1=0.8138 (ep 38)
  Seed 2: acc=0.7870 F1=0.7848 (ep 37)
  Mean acc: 0.8117 +/- 0.0190
  Mean F1:  0.8099 +/- 0.0192
  Majority baseline: acc=0.5463 F1=0.3533
